In [1]:
!pip --version

pip 26.2.1 from D:\in0902\ex0914\.venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
!pip install langchain-community pypdf

In [2]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

FILE_PATH = "../data/SPRI_AI_Brief_2023년12월호_F.pdf"

C:\Users\user\AppData\Local\Temp\ipykernel_15828\2732862613.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1. PDF 로더 (PyPDFLoader)

`load()`는 페이지 단위로 한 번에 읽고, `lazy_load()`는 한 페이지씩 읽어 메모리를 아낍니다.
`FILE_PATH`의 PDF 파일이 실제로 존재해야 실행됩니다.

In [4]:
loader = PyPDFLoader(FILE_PATH)
docs = loader.load()  # 페이지 단위로 Document 리스트 반환

print(len(docs))  # 전체 페이지 수
print(docs[0].page_content[:300])  # 첫 페이지 본문 일부
print(docs[0].metadata)  # {'source': ..., 'page': 0}

23
2023년 12월호
{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': '../data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}


In [5]:
for doc in loader.lazy_load():
    print(doc.metadata["page"], len(doc.page_content))

0 10
1 2006
2 17
3 1585
4 1544
5 1523
6 1364
7 1579
8 1469
9 1395
10 1353
11 1314
12 1253
13 1382
14 1422
15 1440
16 1309
17 1415
18 1570
19 1098
20 1036
21 832
22 87


## 2. 텍스트 파일 (TextLoader)

`../data/sample.txt`는 예시 경로이므로 실제 파일 경로에 맞게 바꿔서 실행하세요.

In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/sample.txt", encoding="utf-8")
docs = loader.load()

RuntimeError: Error loading ../data/sample.txt

## 3. CSV (CSVLoader)

`../data/sample.csv`는 예시 경로이므로 실제 파일 경로에 맞게 바꿔서 실행하세요.

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("../data/sample.csv", encoding="utf-8")
docs = loader.load()  # 행 하나가 Document 하나

## 4. 웹 페이지 (WebBaseLoader)

`beautifulsoup4`가 필요합니다: `%pip install beautifulsoup4`

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://example.com")
docs = loader.load()

## 5. 폴더 안의 여러 파일 (DirectoryLoader)

`glob` 패턴에 맞는 파일을 모두 읽습니다. `../data` 폴더에 PDF가 있어야 동작합니다.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    "../data",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)
docs = loader.load()

## 6. 직접 Document 만들기

로더 없이 `Document`를 직접 만들 수 있습니다. 모든 로더의 결과는 `page_content`(본문)와 `metadata`(출처, 페이지 등)를 가진 `Document` 리스트입니다.

In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="본문 내용",
    metadata={"source": "manual", "page": 1},
)

## 7. Word 문서 (Docx2txtLoader)

`docx2txt`가 필요합니다: `%pip install docx2txt`

`../data/sample.docx`는 예시 경로이므로 실제 파일 경로에 맞게 바꿔서 실행하세요. Word 문서는 페이지 구분이 없어 문서 전체가 `Document` 하나로 반환됩니다.

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("../data/sample.docx")
docs = loader.load()

print(len(docs))
print(docs[0].page_content[:300])
print(docs[0].metadata)  # {'source': ...}

## 8. 다른 PDF 로더 (PyMuPDFLoader)

`pymupdf`가 필요합니다: `%pip install pymupdf`

PyPDFLoader보다 추출 속도가 빠르고, 메타데이터(제목, 작성자, 총 페이지 수 등)가 더 풍부합니다. 사용법은 동일합니다.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()

print(len(docs))
print(docs[0].page_content[:300])
print(docs[0].metadata)  # source, page, total_pages, title, author 등

## 9. 위키피디아 (WikipediaLoader)

`wikipedia`가 필요합니다: `%pip install wikipedia`

검색어에 맞는 문서를 가져옵니다. 인터넷 연결이 필요하며, `lang`으로 언어를, `load_max_docs`로 가져올 문서 수를 지정합니다.

In [ ]:
from langchain_community.document_loaders import WikipediaLoader

loader = WikipediaLoader(query="인공지능", lang="ko", load_max_docs=2)
docs = loader.load()

for doc in docs:
    print(doc.metadata["title"], len(doc.page_content))